## Bag of Words (BoW)

In natural language processing (NLP), the Bag of Words (BoW) model is a simple and commonly used technique for representing text data. Every word in a document is treated as a separate feature, and the frequency of each word is counted to create a vector representation of the document. The order of words is ignored, hence the term "bag" of words.

In [17]:
import pandas as pd

corpus = [
    "the cat sat on the mat and the dog sat on the rug",
    "the cat sat on the hat",
    "the dog lay on the mat",
    "the dog chased the cat",
    "the cat and the dog are friends",
]

In [18]:
vocabulary = sorted(set(word for doc in corpus for word in doc.split()))
print("Vocabulary:", vocabulary)

word_to_index = {word: idx for idx, word in enumerate(vocabulary)}
print("Word to Index Mapping:", word_to_index)

Vocabulary: ['and', 'are', 'cat', 'chased', 'dog', 'friends', 'hat', 'lay', 'mat', 'on', 'rug', 'sat', 'the']
Word to Index Mapping: {'and': 0, 'are': 1, 'cat': 2, 'chased': 3, 'dog': 4, 'friends': 5, 'hat': 6, 'lay': 7, 'mat': 8, 'on': 9, 'rug': 10, 'sat': 11, 'the': 12}


In [19]:
def build_bow_vector(document, word_to_index):
    vector = [0] * len(word_to_index)
    for word in document.split():
        if word in word_to_index:
            vector[word_to_index[word]] += 1
    return vector

In [20]:
bow_matrix_scratch = []
for doc in corpus:
    vec = build_bow_vector(doc, word_to_index)
    bow_matrix_scratch.append(vec)

# Display as a table
df_scratch = pd.DataFrame(bow_matrix_scratch, columns=vocabulary)
df_scratch.index = [f"Doc {i+1}" for i in range(len(corpus))]
print("Word-Document Matrix:")
print(df_scratch.to_string())

Word-Document Matrix:
       and  are  cat  chased  dog  friends  hat  lay  mat  on  rug  sat  the
Doc 1    1    0    1       0    1        0    0    0    1   2    1    2    4
Doc 2    0    0    1       0    0        0    1    0    0   1    0    1    2
Doc 3    0    0    0       0    1        0    0    1    1   1    0    0    2
Doc 4    0    0    1       1    1        0    0    0    0   0    0    0    2
Doc 5    1    1    1       0    1        1    0    0    0   0    0    0    2


##  Term Frequency x Inverse Document Frequency

TF-IDF is a statistical measure used to evaluate the importance of a word in a document relative to a collection of documents (corpus). It combines two metrics: Term Frequency (TF) and Inverse Document Frequency (IDF). Simply put: _A word is important if it appears a lot in one document but rarely across all documents._

In [21]:
def term_frequency(document):
	words = document.split()
	tf = {}
	for word in words:
		tf[word] = tf.get(word, 0) + 1
	
	for word in tf:
		tf[word] /= len(words)
	return tf

In [22]:
import math

# for example the apperans in 3/3 of the doc -> log(3/3) = 0 -> useless word
def inverse_document_frequency(corpus):
	total_docs = len(corpus)
	idf = {}
	all_words = set(word for doc in corpus for word in doc.split())
	for word in all_words:
		doc_count = sum(1 for doc in corpus if word in doc.split())
		idf[word] = math.log(total_docs / doc_count)
	return idf

In [23]:
def if_idf(document, corpus):
	tf = term_frequency(document)
	idf = inverse_document_frequency(corpus)
	tf_idf = {word: tf.get(word, 0) * idf.get(word, 0) for word in idf}
	return tf_idf

In [24]:
print("TF-IDF:")
for i, (doc, tf_idf) in enumerate(zip(corpus, [if_idf(doc, corpus) for doc in corpus])):
	print(f"Document {i+1}: {tf_idf}")

TF-IDF:
Document 1: {'rug': 0.12380291634108465, 'chased': 0.0, 'cat': 0.01716488856263152, 'on': 0.07858855750246012, 'lay': 0.0, 'dog': 0.01716488856263152, 'friends': 0.0, 'and': 0.0704839024518581, 'are': 0.0, 'mat': 0.0704839024518581, 'the': 0.0, 'sat': 0.1409678049037162, 'hat': 0.0}
Document 2: {'rug': 0.0, 'chased': 0.0, 'cat': 0.037190591885701625, 'on': 0.08513760396099845, 'lay': 0.0, 'dog': 0.0, 'friends': 0.0, 'and': 0.0, 'are': 0.0, 'mat': 0.0, 'the': 0.0, 'sat': 0.15271512197902584, 'hat': 0.26823965207235}
Document 3: {'rug': 0.0, 'chased': 0.0, 'cat': 0.0, 'on': 0.08513760396099845, 'lay': 0.26823965207235, 'dog': 0.037190591885701625, 'friends': 0.0, 'and': 0.0, 'are': 0.0, 'mat': 0.15271512197902584, 'the': 0.0, 'sat': 0.0, 'hat': 0.0}
Document 4: {'rug': 0.0, 'chased': 0.3218875824868201, 'cat': 0.04462871026284196, 'on': 0.0, 'lay': 0.0, 'dog': 0.04462871026284196, 'friends': 0.0, 'and': 0.0, 'are': 0.0, 'mat': 0.0, 'the': 0.0, 'sat': 0.0, 'hat': 0.0}
Document 5: 

## [Word2Vec](https://www.geeksforgeeks.org/python/python-word-embedding-using-word2vec/)
Each word is represented by a vector (with real values) of fixed size (e.g. 100, 200, 300) that contains information about its semantics.

In [25]:
from gensim.models import Word2Vec

### Continuous bag of words (CBOW)
> Continuous Bag of Words (CBOW) is a popular natural language processing technique used to generate word embeddings. Word embeddings are important for many NLP tasks because they capture semantic and syntactic relationships between words in a language. CBOW is a neural network-based algorithm that predicts a target word given its surrounding context words. It is a type of "unsupervised" learning, meaning that it can learn from unlabeled data, and it is often used to pre-train word embeddings that can be used for various NLP tasks such as sentiment analysis, text classification, and machine translation. 
### Skip-gram
> In skip-gram architecture of word2vec, the input is the center word and the predictions are the context words. Consider an array of words W, if W(i) is the input (center word), then W(i-2), W(i-1), W(i+1), and W(i+2) are the context words if the sliding window size is 2.

In [26]:
tokenized_corpus = [sentence.split() for sentence in corpus]

model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=10,
    window=2,         # how many words left/right to consider as context
    min_count=1,      # ignore words that appear less than N times
    sg=0,             # 0 = CBOW, 1 = Skip-gram
    epochs=200,       # training iterations
)

In [27]:
print(f"  Vocabulary size : {len(model.wv)}")
print(f"  Vector size     : {model.vector_size}")
print(f"  Words           : {list(model.wv.index_to_key)}")

  Vocabulary size : 13
  Vector size     : 10
  Words           : ['the', 'dog', 'on', 'cat', 'sat', 'and', 'mat', 'friends', 'are', 'chased', 'lay', 'hat', 'rug']


In [28]:
for word in ["cat", "dog", "mat", "hat"]:
    print(f"\n  Vector for '{word}':")
    print(f"  {model.wv[word].round(3)}")


  Vector for 'cat':
  [-0.08  -0.002  0.103 -0.068 -0.025 -0.034  0.1   -0.041 -0.029 -0.056]

  Vector for 'dog':
  [ 0.067 -0.005 -0.036  0.075 -0.052 -0.041  0.059  0.04  -0.131 -0.11 ]

  Vector for 'mat':
  [-0.021  0.01  -0.034 -0.073 -0.017  0.011  0.008  0.073 -0.054  0.015]

  Vector for 'hat':
  [-0.005 -0.082 -0.055  0.072  0.032  0.069  0.072  0.08  -0.044 -0.006]


Find similar words.

In [29]:
for word in ["cat", "dog", "mat"]:
    similar = model.wv.most_similar(word, topn=3)
    print(f"\n  Words most similar to '{word}':")
    for similar_word, score in similar:
        print(f"    '{similar_word}' -> similarity: {score:.3f}")


  Words most similar to 'cat':
    'friends' -> similarity: 0.375
    'the' -> similarity: 0.338
    'lay' -> similarity: 0.335

  Words most similar to 'dog':
    'the' -> similarity: 0.735
    'on' -> similarity: 0.547
    'hat' -> similarity: 0.405

  Words most similar to 'mat':
    'sat' -> similarity: 0.610
    'rug' -> similarity: 0.390
    'friends' -> similarity: 0.312


Similarity between two words.

In [30]:
pairs = [("cat", "dog"), ("cat", "mat"), ("cat", "hat")]
for w1, w2 in pairs:
    score = model.wv.similarity(w1, w2)
    print(f"  similarity('{w1}', '{w2}') = {score:.3f}")

  similarity('cat', 'dog') = 0.066
  similarity('cat', 'mat') = 0.068
  similarity('cat', 'hat') = -0.203


## GloVe (Global Vectors for Word Representation)
GloVe is a word embedding technique that generates vector representations of words based on their co-occurrence in a large corpus of text. GloVe works by constructing a co-occurrence matrix that counts how often each word appears in the context of other words, and then factorizing this matrix to obtain dense vector representations for each word.

In [32]:
import gensim.downloader as api

glove_model = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


In [33]:
print(glove_model.most_similar("cat"))
print(glove_model.similarity("cat", "dog"))

[('dog', 0.9218006134033203), ('rabbit', 0.8487821221351624), ('monkey', 0.8041081428527832), ('rat', 0.7891963124275208), ('cats', 0.7865270972251892), ('snake', 0.7798910737037659), ('dogs', 0.7795814871788025), ('pet', 0.7792249917984009), ('mouse', 0.773166835308075), ('bite', 0.7728800177574158)]
0.9218005


## N-grams
An n-gram is a contiguous sequence of n items from a given sample of text or speech. The items can be phonemes, syllables, letters, words, or base pairs according to the application. The n-grams typically are collected from a text or speech corpus. When the items are words, n-grams may also be called shingles.

In [35]:
def n_grams(text, n):
    words = text.split()
    ngrams = []
    for i in range(len(words) - n + 1):
        ngram = " ".join(words[i:i+n])
        ngrams.append(ngram)
    return ngrams

In [36]:
def build_vocab(corpus, n):
    vocab = sorted(set(
        ngram
        for doc in corpus
        	for ngram in n_grams(doc, n)
    ))
    return vocab

In [37]:
for n, name in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    vocab = build_vocab(corpus, n)
    print(f"\n{name} vocabulary ({len(vocab)} tokens):")
    print(f"  {vocab}")


Unigrams vocabulary (13 tokens):
  ['and', 'are', 'cat', 'chased', 'dog', 'friends', 'hat', 'lay', 'mat', 'on', 'rug', 'sat', 'the']

Bigrams vocabulary (18 tokens):
  ['and the', 'are friends', 'cat and', 'cat sat', 'chased the', 'dog are', 'dog chased', 'dog lay', 'dog sat', 'lay on', 'mat and', 'on the', 'sat on', 'the cat', 'the dog', 'the hat', 'the mat', 'the rug']

Trigrams vocabulary (21 tokens):
  ['and the dog', 'cat and the', 'cat sat on', 'chased the cat', 'dog are friends', 'dog chased the', 'dog lay on', 'dog sat on', 'lay on the', 'mat and the', 'on the hat', 'on the mat', 'on the rug', 'sat on the', 'the cat and', 'the cat sat', 'the dog are', 'the dog chased', 'the dog lay', 'the dog sat', 'the mat and']


The N-gram matrix is like the BoW matrix, but instead of counting single words, it counts sequences of n words. For example, if n=2 (bigrams), the matrix would count how many times each pair of consecutive words appears in the document. This can capture some of the context and word order that BoW misses.

In [39]:
def build_ngram_matrix(corpus, n):
    vocab = build_vocab(corpus, n)
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    matrix = []
    for doc in corpus:
        vector = [0] * len(vocab)
        for ngram in n_grams(doc, n):
            vector[word_to_idx[ngram]] += 1
        matrix.append(vector)
    return matrix, vocab

In [49]:
matrix, vocab = build_ngram_matrix(corpus, 3) # build trigram matrix
print(f"\nTrigrams (n=3):")
print(f"  {'Doc':<7}", end="")
for v in vocab:
	print(f"{v:<18}", end="")
print()
for i, vec in enumerate(matrix):
	print(f"  Doc {i+1}  ", end="")
	for val in vec:
		print(f"{val:<18}", end="")
	print()


Trigrams (n=3):
  Doc    and the dog       cat and the       cat sat on        chased the cat    dog are friends   dog chased the    dog lay on        dog sat on        lay on the        mat and the       on the hat        on the mat        on the rug        sat on the        the cat and       the cat sat       the dog are       the dog chased    the dog lay       the dog sat       the mat and       
  Doc 1  1                 0                 1                 0                 0                 0                 0                 1                 0                 1                 0                 1                 1                 2                 0                 1                 0                 0                 0                 1                 1                 
  Doc 2  0                 0                 1                 0                 0                 0                 0                 0                 0                 0                 1                 